<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/09-user-defined-functions/03-arrow-optimization.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 9.3 — Arrow: the two switches, timed

Switch 1 (driver-side conversion for toPandas/createDataFrame) and
switch 2 (Arrow transport for plain UDFs) optimize different
boundaries. We time both, shrink the shared batch-size dial, and
poke the timezone seam.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("9.3-arrow-optimization")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

orders = (
    spark.read.csv(f"{DATA_DIR}/orders.csv", header=True, inferSchema=True)
    .withColumn("revenue", F.round(col("quantity") * col("unit_price"), 2))
)
orders.select("order_id", "customer_id", "quantity", "revenue").show(3)

## Where does YOUR session stand?

Defaults have moved across Spark versions — print, don't assume.
Production scripts set these explicitly either way.

In [ ]:
for k in ["spark.sql.execution.arrow.pyspark.enabled",
          "spark.sql.execution.arrow.pyspark.fallback.enabled",
          "spark.sql.execution.arrow.maxRecordsPerBatch",
          "spark.sql.execution.pythonUDF.arrow.enabled"]:
    print(f"{k} = {spark.conf.get(k)}")

## Switch 1: toPandas() with and without Arrow

Same collect, two transports: row-by-row pickle assembly vs
columnar batches pandas adopts nearly zero-copy.

In [ ]:
import time

conv = spark.range(1_000_000).select(
    "id", (col("id") % 97).alias("bucket"), F.rand(42).alias("x"))

def time_topandas(flag):
    spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", flag)
    start = time.perf_counter()
    pdf = conv.toPandas()
    print(f"arrow={flag:>5}: {time.perf_counter() - start:5.2f} s "
          f"({len(pdf):,} rows)")

time_topandas("false")
time_topandas("true")
# Arrow made it FASTER, not SAFER: all 1M rows still sit in driver
# memory (3.6's rule, unrepealed).

## The silent fallback

Types Arrow can't convert (version-dependent — maps and nested
structs historically) don't fail: with fallback enabled, Spark
quietly takes the slow path and only a log line tells you. Flip it
to false in test suites so regressions are loud.

In [ ]:
spark.conf.set("spark.sql.execution.arrow.pyspark.fallback.enabled", "false")
tricky = orders.select("order_id",
                       F.create_map(F.lit("qty"), col("quantity")).alias("m"))
try:
    tricky.toPandas()
    print("your Spark converts MapType via Arrow — no fallback needed")
except Exception as e:
    print("loud failure (would have been a silent slow path):",
          str(e).splitlines()[0])
spark.conf.set("spark.sql.execution.arrow.pyspark.fallback.enabled", "true")

## Switch 2: the three-way UDF race

Arrow-optimized plain UDFs (3.5+): same per-row Python calls, Arrow
transport instead of pickle. Reads as "toll #1 refunded, toll #2
still due" — which the times should show, with pandas_udf ahead of
both.

In [ ]:
import pandas as pd
from pyspark.sql.functions import pandas_udf

def bench(label, df):
    start = time.perf_counter()
    df.write.format("noop").mode("overwrite").save()
    print(f"{label:>16}: {time.perf_counter() - start:5.2f} s")

fn = lambda x: (x % 97) * 1.5 + 1.0
pickled = F.udf(fn, returnType="double")
arrowed = F.udf(fn, returnType="double", useArrow=True)

@pandas_udf("double")
def vectorized(s: pd.Series) -> pd.Series:
    return (s % 97) * 1.5 + 1.0

big = spark.range(2_000_000)
bench("pickled udf", big.select(pickled("id")))
bench("arrow udf",   big.select(arrowed("id")))
bench("pandas udf",  big.select(vectorized("id")))

## The shared dial: maxRecordsPerBatch

One Arrow batch = one pandas-UDF call = one unit of Python-worker
memory. Watch the batches your UDF actually receives change size.

In [ ]:
@pandas_udf("long")
def batch_len(s: pd.Series) -> pd.Series:
    return pd.Series([len(s)] * len(s))

fifty_k = spark.range(50_000, numPartitions=2)   # 25k rows per partition

print("default (10000):")
fifty_k.select(batch_len("id").alias("n")).distinct().show()

spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "2000")
print("maxRecordsPerBatch=2000:")
fifty_k.select(batch_len("id").alias("n")).distinct().show()

spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "10000")
# smaller batches: less worker memory, more per-batch overhead.
# It's the FIRST dial for Python-worker OOMs, before executor memory.

## The timezone seam

The stored instant never changes; the wall-clock value pandas
receives follows spark.sql.session.timeZone. Two sessions, two
different-looking "same" timestamps — pin the zone in production.

In [ ]:
ts = spark.sql("SELECT timestamp'2026-07-09 12:00:00' AS ts")

spark.conf.set("spark.sql.session.timeZone", "UTC")
print("UTC session:         ", ts.toPandas()["ts"][0])

spark.conf.set("spark.sql.session.timeZone", "Asia/Kolkata")
print("Asia/Kolkata session:", ts.toPandas()["ts"][0])

spark.conf.set("spark.sql.session.timeZone", "UTC")

## Exercises

1. Re-run the toPandas race at 100k and 5M rows. Where does the
   Arrow advantage grow fastest, and why does that follow from
   "per-row cost vs per-batch cost"?
2. `createDataFrame(pandas_df)` is switch 1's other half. Build a
   1M-row pandas frame and time the round trip both ways.
3. From the three-way race: estimate what fraction of the pickled
   UDF's cost was transport (pickle) vs interpreter calls. Which
   number would a `useArrow=True` migration of a legacy codebase
   improve, and which UDFs would see almost nothing?
4. Set maxRecordsPerBatch to 100 and re-run the race cell. Explain
   the change in the pandas UDF's time.
5. A nightly job reads timestamps, does pandas-UDF feature
   engineering, and writes back. It runs from two schedulers, one
   configured UTC, one Asia/Kolkata. Describe the bug that ships,
   and the one-line config that prevents it.

In [ ]:
# your exercise code here